# Remove ripple from every TIFF in a session folder

Detect the moving periodic banding (PMT ripple) in each ScanImage TIFF of a
session, write cleaned copies (plain BigTIFF, same dtype and shape) plus a JSON
report, and inspect the result.

Run from `notebooks/` after `pip install -e .`. The same pipeline is available
from the shell as `bandaid-clean <session_dir> --diagnostics`.

In [ ]:
import json
import logging
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from bandaid import ScanGeometry, detect_ripple, remove_ripple
from bandaid.diagnostics import (
    plot_fit_tracks,
    plot_frame_comparison,
    plot_profiles_over_time,
    plot_tspace_spectra,
)
from bandaid.io import (
    get_files,
    load_tiff_stack,
    read_scanimage_metadata,
    save_session_metadata,
    write_json,
)
from bandaid.ripple import RowGroups, column_profiles

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")
plt.rcParams["figure.dpi"] = 110

## Configuration

In [ ]:
data_dir = Path("/Volumes/group/douglgrp/HaKu/data/mini2p/HK13/HK13_20260814")
if not data_dir.is_dir():  # fall back to the local example session
    data_dir = Path("./data/HK13/HK13_20260814").resolve()
    print(f"network volume not mounted; using {data_dir}")

out_dir = data_dir / "cleaned"      # cleaned TIFFs and ripple_report.json go here
n_harmonics = 8                     # harmonics in the ripple model
overwrite = False                   # re-process files whose output already exists
copy_if_clean = False               # copy files without detectable ripple unchanged
write_session_metadata = True       # also write <data_dir>/session_metadata.json

detections, reports = {}, {}
print("input :", data_dir)
print("output:", out_dir)

## Files in the session

In [ ]:
tiffs = [Path(p) for p, _ in get_files(data_dir)]
print(f"{len(tiffs)} TIFF(s)")
for p in tiffs:
    meta = read_scanimage_metadata(p, frames=False)
    print(
        f"  {p.name:40s} {meta['n_frames']:6d} frames  {meta['shape']}  {meta['dtype']}  "
        f"{meta['n_channels']} ch  {p.stat().st_size / 1e6:6.0f} MB"
    )

## Detection only

Reads three blocks of frames per file and reports the ripple's harmonic index
relative to the line rate, its SNR, how many harmonics are visible, its
amplitude in counts and how fast the pattern crawls (cycles per frame).

In [ ]:
hdr = f"{'file':40s} {'ripple':7s} {'n':>4s} {'f0 kHz':>7s} {'SNR dB':>7s} {'harm':>4s} {'amp':>6s} {'cyc/frame':>9s}"
print(hdr)
print("-" * len(hdr))
for p in tiffs:
    det = detect_ripple(p)
    detections[p.name] = det
    print(
        f"{p.name[:40]:40s} {'yes' if det.detected else 'no':7s} {det.harmonic_index!s:>4s} "
        f"{det.f0_hz / 1e3:7.1f} {det.snr_db:7.1f} {det.n_harmonics_detected:4d} "
        f"{det.amplitude_counts:6.1f} {det.phase_advance_cycles_per_frame:9.4f}"
    )

## Process every file

Each file is streamed twice (profiles, then subtraction), so memory does not
depend on movie length. Files without detectable ripple are skipped unless
`copy_if_clean` is set. Existing outputs are skipped unless `overwrite` is set.

In [ ]:
out_dir.mkdir(parents=True, exist_ok=True)
for p in tiffs:
    out = out_dir / p.name
    if out.exists() and not overwrite:
        print(f"skip {p.name}: {out.name} exists (set overwrite = True to redo)")
        continue
    t0 = time.time()
    _, report = remove_ripple(
        p, out=out, n_harmonics=n_harmonics, detection=detections.get(p.name), copy_if_clean=copy_if_clean
    )
    reports[p.name] = report
    if report.removed:
        m = report.metrics[0]
        print(
            f"{p.name}: cleaned in {time.time() - t0:.1f} s; worst harmonic line after = "
            f"{max(m['line_power_ratio_after'].values()):.2f}; mean-image change = {m['mean_image_change_max']:.3f}"
        )
    else:
        print(f"{p.name}: no ripple detected, nothing written ({time.time() - t0:.1f} s)")

report_path = out_dir / "ripple_report.json"
if reports:
    write_json(
        {"session": {"session_dir": str(data_dir), "n_files": len(tiffs)}, "files": [r.to_dict() for r in reports.values()]},
        report_path,
    )
    print("report:", report_path)
if write_session_metadata:
    save_session_metadata(data_dir)

## Results overview (from the report, so it also works in a later session)

In [ ]:
rep = json.loads(report_path.read_text())
hdr = f"{'file':40s} {'removed':8s} {'amp':>6s} {'rho 1e-4':>9s} {'worst line':>10s} {'mean chg':>9s} {'outliers':>8s} {'time s':>7s}"
print(hdr)
print("-" * len(hdr))
for f in rep["files"]:
    name = Path(f["source"]).name
    if not f["removed"]:
        print(f"{name[:40]:40s} {'no':8s}")
        continue
    fit, m = f["fit"][0], f["metrics"][0]
    print(
        f"{name[:40]:40s} {'yes':8s} {fit['amplitude_median_counts'][0]:6.1f} "
        f"{fit['rho_cycles_per_row_median'] * 1e4:9.2f} {max(m['line_power_ratio_after'].values()):10.2f} "
        f"{m['mean_image_change_max']:9.3f} {fit['n_outlier_frames']:8d} {f['runtime_s']['total']:7.1f}"
    )

## Inspect one file

Spectrum of the column profiles in scan time before and after (the dashed lines
are the fitted harmonics), the crawling stripes over time, a raw/cleaned/removed
frame, and, if the file was processed in this session, the per-frame fit tracks.

In [ ]:
name = tiffs[0].name          # pick a file
src, out = data_dir / name, out_dir / name
meta = read_scanimage_metadata(src, frames=False)
geometry = ScanGeometry.from_si_params(meta["si_params"])
a = max(0, meta["n_frames"] // 2 - 150)
b = min(meta["n_frames"], a + 300)
raw = load_tiff_stack(src, frames=range(a, b))
cleaned = load_tiff_stack(out, frames=range(a, b))

groups = RowGroups.from_geometry(geometry, 1)
before = column_profiles(raw, groups)[:, 0]
after = column_profiles(cleaned, groups)[:, 0]
report = reports.get(name)
if report is not None:
    f0 = report.detection[0]["f0_hz"]
elif name in detections:
    f0 = detections[name].f0_hz
else:
    f0 = detect_ripple(src).f0_hz

fig, axes = plt.subplots(2, 1, figsize=(10, 8))
plot_tspace_spectra(before, after, geometry, f0, n_harmonics, ax=axes[0])
plot_profiles_over_time(before[:200], ax=axes[1], title=f"dynamic column profiles, frames {a}..{a + 200}")
fig.tight_layout()

fig, axes = plt.subplots(1, 3, figsize=(13, 4.3))
plot_frame_comparison(raw[150], cleaned[150], title=f"{name}: frame {a + 150}", axes=axes)
fig.tight_layout()

if report is not None and report.fits and report.fits[0] is not None:
    fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
    plot_fit_tracks(report.fits[0], geometry.frame_period_s, axes=axes)
    fig.tight_layout()

## In-memory preview of a frame range (writes nothing)

Useful for trying parameters (`n_harmonics`, `fit_mask`, `window_frames`) on a
short block before processing the whole session.

In [ ]:
path = tiffs[0]
frames = range(0, 600)
geometry = ScanGeometry.from_si_params(read_scanimage_metadata(path, frames=False)["si_params"])
mov = load_tiff_stack(path, frames=frames)
cleaned_block, block_report = remove_ripple(mov, geometry, n_harmonics=n_harmonics)
print("line ratios after:", {k: round(v, 2) for k, v in block_report.metrics[0]["line_power_ratio_after"].items()})
print("dynamic variance removed:", round(block_report.metrics[0]["dynamic_variance_removed"], 3))

fig, axes = plt.subplots(1, 3, figsize=(13, 4.3))
plot_frame_comparison(mov[300], cleaned_block[300], title=f"{path.name}: frame {frames.start + 300} (in-memory block)", axes=axes)
fig.tight_layout()